# 072 · Why static embeddings are not enough

The lesson claims a static embedding encodes a word's **average** usage, so the
majority sense takes the vector and the minority sense leaves no trace.

That is measurable, and this notebook measures it: build embeddings from raw
co-occurrence, then check what happened to an ambiguous word.

Needs `scikit-learn` and `numpy`. Downloads the 20 newsgroups corpus (~14 MB)
on first run.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

docs = fetch_20newsgroups(subset="train",
                          remove=("headers", "footers", "quotes")).data
print(f"{len(docs):,} documents")

## Build embeddings from co-occurrence alone

No dictionary, no labels, no supervision — only which words appear in the same
document. PPMI weighting, then SVD to 60 dimensions. This is how embeddings
worked before word2vec, and it is enough to make the point.

In [ ]:
counts = CountVectorizer(max_features=4000, stop_words="english", min_df=5)
X = counts.fit_transform(docs)
vocab = np.array(counts.get_feature_names_out())
index = {w: i for i, w in enumerate(vocab)}

binary  = (X > 0).astype(np.float32)
cooccur = (binary.T @ binary).toarray()
np.fill_diagonal(cooccur, 0)

total    = cooccur.sum()
marginal = cooccur.sum(1) / total
with np.errstate(divide="ignore", invalid="ignore"):
    ppmi = np.maximum(0, np.log((cooccur / total) / np.outer(marginal, marginal) + 1e-12))

U, S, _ = np.linalg.svd(ppmi, full_matrices=False)
E = U[:, :60] * S[:60]
E /= np.linalg.norm(E, axis=1, keepdims=True) + 1e-9
print(f"embeddings: {E.shape[0]:,} words x {E.shape[1]}")

In [ ]:
def neighbours(word, k=8):
    sims = E @ E[index[word]]
    sims[index[word]] = -9
    return [vocab[j] for j in np.argsort(-sims)[:k]]

for w in ("space", "car", "medical", "windows"):
    print(f"  {w:<9} -> {', '.join(neighbours(w, 5))}")

Embeddings genuinely work — nothing told the model those words were related.
Now the problem.

## What happened to an ambiguous word

`drive` means a storage device in the `comp.*` groups and a vehicle action in
`rec.autos`. Both senses are present in this corpus. Count them.

In [ ]:
STORAGE = {"disk", "scsi", "hard", "controller", "floppy", "ide",
           "motherboard", "drives", "boot", "pin"}
VEHICLE = {"car", "engine", "miles", "road", "speed", "cars",
           "tires", "driving", "brake"}

word = "drive"
ti = index[word]
present = np.asarray(binary[:, ti].todense()).ravel() > 0

s_hits = np.asarray(binary[:, [index[w] for w in STORAGE if w in index]].sum(1)).ravel()
v_hits = np.asarray(binary[:, [index[w] for w in VEHICLE if w in index]].sum(1)).ravel()

storage = int((present & (s_hits > v_hits) & (s_hits > 0)).sum())
vehicle = int((present & (v_hits > s_hits) & (v_hits > 0)).sum())

print(f"'{word}' occurs in {int(present.sum())} documents")
print(f"  storage sense: {storage:>4}  ({storage/(storage+vehicle):.0%})")
print(f"  vehicle sense: {vehicle:>4}  ({vehicle/(storage+vehicle):.0%})")

In [ ]:
nb8 = neighbours(word, 8)
print(f"eight nearest neighbours of '{word}':")
print(f"  {', '.join(nb8)}")
print()
print(f"  storage-sense: {sum(n in STORAGE for n in nb8)}/8")
print(f"  vehicle-sense: {sum(n in VEHICLE for n in nb8)}/8")
print()
print(f"{vehicle} documents used the vehicle sense.")
print("Check how much of it survived in the vector.")

## Read the result carefully

The minority sense was not *blended* with the majority — it was **replaced**.
And that one vector is what every downstream layer receives, including for a
sentence like *the drive to work*.

This is not the embedding failing. Representing average usage is exactly its
job. **The job is the problem**, and self-attention exists to replace it with
a representation computed from the sentence a word appears in.

## Exercise

1. Try `memory` (RAM vs recollection), `bus` (hardware vs vehicle), `space`.
   Which are genuinely ambiguous *in this corpus* and which are not?
2. Construct a corpus subset where the vehicle sense is the majority. Rebuild
   the embeddings. Does the neighbourhood flip? What does that tell you about
   how much the vector says about *language* versus about *your data*?
3. `orange` near `Apple` should not pull towards fruit. Why can no scheme that
   simply averages a context window achieve that, and what does it imply about
   the mechanism self-attention needs?